BASELINE COMBINATION

* C3-CZ vs C3-P3
* activity+mobility+band powers

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy import signal
from scipy.integrate import simpson

DATA = Path('eeg_starter/data')
FS = 64
montage = ['FP1-F7', 'F7-T3', 'T3-T5', 'T5-O1', 'FP2-F8', 'F8-T4',
           'T4-T6',  'T6-O2', 'T3-C3', 'C3-CZ',  'CZ-C4',
           'C4-T4',  'FP1-F3','F3-C3', 'C3-P3',  'P3-O1',
           'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2']
CHANNELS = ['C3-CZ', 'C3-P3']

WIN_S = 30
WIN = WIN_S * FS
N_WIN = 300 // WIN_S

BANDS = {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 12), 'beta': (12, 30)}
FEATURES = ['log activity', 'mobility'] + [f'log {b}' for b in BANDS]

train_seg = torch.load(DATA / 'segments_h1k_train.pt')
val_seg = torch.load(DATA / 'segments_h1k_val.pt')
age_train = np.array([s[3] for s in train_seg]); age_val = np.array([s[3] for s in val_seg])
y_train, y_val = np.log(age_train), np.log(age_val)
print(f'train {len(train_seg)}, val {len(val_seg)} subjects; channels', {c: montage.index(c) for c in CHANNELS})

train 825, val 103 subjects; channels {'C3-CZ': 9, 'C3-P3': 14}


In [ ]:
WELCH_S = 4

def hjorth(w):
    dx = np.diff(w, axis=1)
    var_x, var_dx = w.var(axis=1), dx.var(axis=1)
    flat = var_x < 1e-6
    with np.errstate(divide='ignore', invalid='ignore'):
        act, mob = np.log(var_x), np.sqrt(var_dx / var_x)
    act[flat] = np.nan; mob[flat] = np.nan
    return act, mob

def log_band_powers(w):
    freqs, psd = signal.welch(w, FS, nperseg=WELCH_S * FS, axis=1)
    df = freqs[1] - freqs[0]
    out = []
    for low, high in BANDS.values():
        idx = (freqs >= low) & (freqs <= high)
        p = simpson(psd[:, idx], dx=df, axis=1)
        with np.errstate(divide='ignore'):
            out.append(np.where(p > 0, np.log(np.maximum(p, 1e-30)), np.nan))
    return out

def subject_features(segments, ch_name):
    ch = montage.index(ch_name)
    X = np.full((len(segments), len(FEATURES)), np.nan)
    for i, s in enumerate(segments):
        eeg = torch.load(DATA / 'pre_processed' / s[4].replace('.edf', '.pt'))
        w = eeg[ch, :N_WIN * WIN].double().numpy().reshape(N_WIN, WIN)
        per_window = np.stack([*hjorth(w), *log_band_powers(w)], axis=1)
        X[i] = np.nanmedian(per_window, axis=0)
    return X

feat = {ch: {'train': subject_features(train_seg, ch), 'val': subject_features(val_seg, ch)} for ch in CHANNELS}
for ch in CHANNELS:
    print(ch, 'train', feat[ch]['train'].shape, 'val', feat[ch]['val'].shape,
          'undefined values:', int(np.isnan(feat[ch]['train']).sum() + np.isnan(feat[ch]['val']).sum()))

C3-CZ train (825, 6) val (103, 6) undefined values: 0
C3-P3 train (825, 6) val (103, 6) undefined values: 0


In [ ]:
def corr(a, b):
    ok = ~np.isnan(a) & ~np.isnan(b)
    return np.corrcoef(a[ok], b[ok])[0, 1]

r_table = pd.DataFrame({ch: [corr(feat[ch]['train'][:, f], y_train) for f in range(len(FEATURES))] for ch in CHANNELS}, index=FEATURES)
r_table.columns = [f'r with log(age), {ch}' for ch in CHANNELS]
r_table.round(3)

,"r with log(age), C3-CZ","r with log(age), C3-P3"
log activity,-0.681,-0.720
mobility,0.717,0.768
log delta,-0.747,-0.780
log theta,-0.559,-0.607
log alpha,-0.147,0.002
log beta,-0.122,-0.043


hmmmmmm P3 the wave?

In [ ]:
def rmse(a, b): return np.sqrt(np.mean((a - b) ** 2))
def r2(y, p):   return 1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)

def fit_predict(Xtr, ytr, Xva):
    mu, sd = Xtr.mean(axis=0), Xtr.std(axis=0)
    D = lambda X: np.column_stack([np.ones(len(X)), (X - mu) / sd])
    theta, *_ = np.linalg.lstsq(D(Xtr), ytr, rcond=None)
    return D(Xtr) @ theta, D(Xva) @ theta

SETS = {'Hjorth (2)': [0, 1], 'Band power (4)': [2, 3, 4, 5], 'Hjorth + band power (6)': [0, 1, 2, 3, 4, 5]}

def row(model, p_tr, p_va):
    a_tr, a_va = np.exp(p_tr), np.exp(p_va)
    return {'model': model, 'train RMSE (y)': rmse(a_tr, age_train), 'val RMSE (y)': rmse(a_va, age_val), 'val MAE (y)': np.mean(np.abs(a_va - age_val)), 'val R^2 (log age)': r2(y_val, p_va), 'val R^2 (age)': r2(age_val, a_va)}

rows = [row('mean of training log(age)', np.full(len(y_train), y_train.mean()), np.full(len(y_val), y_train.mean()))]
preds = {}
for ch in CHANNELS:
    for name, cols in SETS.items():
        p_tr, p_va = fit_predict(feat[ch]['train'][:, cols], y_train, feat[ch]['val'][:, cols])
        preds[(ch, name)] = np.exp(p_va)
        rows.append(row(f'{ch}: {name}', p_tr, p_va))

results = pd.DataFrame(rows).set_index('model').round(3)
results

,train RMSE (y),val RMSE (y),val MAE (y),val R^2 (log age),val R^2 (age)
model,,,,,
mean of training log(age),4.720,4.139,2.913,-0.046,-0.085
C3-CZ: Hjorth (2),2.847,2.178,1.634,0.532,0.700
C3-CZ: Band power (4),2.839,2.028,1.438,0.654,0.740
C3-CZ: Hjorth + band power (6),2.880,1.973,1.370,0.677,0.754
C3-P3: Hjorth (2),2.679,2.293,1.391,0.649,0.667
C3-P3: Band power (4),2.560,1.833,1.196,0.777,0.787
C3-P3: Hjorth + band power (6),2.218,1.623,1.123,0.788,0.833


new baseline, RMSE 1.623

well use C3-P3 with hjorth1+2 and the band powers